# Exercise 1: Workflow Observability with Phoenix

Without observability, multi-agent workflows are black boxes. NAT has a **native Phoenix integration** - one YAML block enables full tracing of every LLM call, tool invocation, and agent step.

No manual instrumentation needed. No code changes. Just config.

## Step 1: Launch Phoenix

Phoenix provides a web UI for exploring traces.

In [ ]:
import os
import phoenix as px

# In Docker: Phoenix runs as a separate service, no launch needed.
# Locally: launch Phoenix in-process.
phoenix_url = os.getenv("PHOENIX_COLLECTOR_ENDPOINT", "").replace("/v1/traces", "")
if not phoenix_url:
    session = px.launch_app()
    phoenix_url = session.url
    print(f"Phoenix launched: {phoenix_url}")
else:
    print(f"Phoenix service: {phoenix_url}")

print("Open this URL in your browser to view traces.")

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

## Step 2: Review the Config

Compare this to the Part 2 config. The **only difference** is the `general.telemetry` section at the bottom:

In [ ]:
with open("config.yaml") as f:
    print(f.read())

That's it. NAT's Phoenix plugin (`nvidia-nat-phoenix`) handles the rest:
- Intercepts every workflow step, tool call, and agent action
- Records inputs, outputs, and timing
- Exports traces to Phoenix via OpenTelemetry

No `OpenAIInstrumentor`, no manual setup. Just YAML.

## Step 3: Run a Workflow with Tracing

In [ ]:
!nat run --config_file config.yaml --input "Kim była Maria Skłodowska-Curie i jakie były jej główne osiągnięcia naukowe?"

## Step 4: Explore Traces in Phoenix UI

Open the Phoenix UI and explore:

1. **Traces view** - See all captured workflow runs
2. **Click a trace** - Expand to see workflow spans, tool calls, inputs/outputs
3. **Compare traces** - Which query took the longest? Which used the most tool calls?

Each trace shows:
- `<workflow>` span - the full workflow execution
- `wiki_search` spans - each tool invocation with input query and output content
- Timing for each step

## Step 5: Run More Queries to Build Up Trace Data

In [ ]:
queries = [
    "Czym jest Kopalnia Soli w Wieliczce?",
    "Kim był Fryderyk Chopin?",
    "Z czego znana jest Stocznia Gdańska?",
]

for i, query in enumerate(queries, 1):
    print(f"\n{'='*60}")
    print(f"Query {i}/{len(queries)}: {query}")
    print(f"{'='*60}")
    !nat run --config_file config.yaml --input "{query}"

## Step 6: Query Traces Programmatically

In [ ]:
from phoenix.client import Client

phoenix_client = Client(base_url=phoenix_url or "http://localhost:6006")
df = phoenix_client.spans.get_spans_dataframe(project_name="bielik-nest-workshop")

print(f"Total spans captured: {len(df)}")
print()

# Show each span
for _, row in df.iterrows():
    name = row.get('name', '?')
    kind = row.get('attributes.openinference.span.kind', '?')
    inp = str(row.get('attributes.input.value', ''))[:60]
    out = str(row.get('attributes.output.value', ''))[:60]
    
    # Calculate duration
    duration = ''
    if 'start_time' in df.columns and 'end_time' in df.columns:
        try:
            dt = (row['end_time'] - row['start_time']).total_seconds() * 1000
            duration = f' ({dt:.0f}ms)'
        except:
            pass
    
    print(f"[{kind}] {name}{duration}")
    print(f"  In:  {inp}...")
    print(f"  Out: {out}...")
    print()

## Bonus: File-Based Tracing (No Phoenix Needed)

NAT also supports writing traces to local files - useful for CI/CD or environments without Phoenix:

```yaml
general:
  telemetry:
    tracing:
      file_traces:
        _type: file
        project: bielik-workshop
        output_path: ./traces
```

Other supported exporters: `langfuse`, `langsmith`, `otelcollector`, `weave`

**Next:** Exercise 2 - Cost tracking with Token Factory pricing model